In [1]:
%pip install -q openai matplotlib numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import nbimport

import os, sys, time, json, re
from tqdm import tqdm

import numpy as np
import matplotlib.pyplot as plt

import pandas as pd

In [3]:
import openai

API_KEY = "j1U_9qgj42eqSHzUBAPaAv5mHNMf9I-bKQA7KJCjDhk"

In [4]:
client = openai.OpenAI(
    api_key=API_KEY,
    base_url="https://api.poe.com/v1",
)

# if __name__ == "__main__":
#     models = client.models.list()
#     print(", ".join([model.id for model in models.data]))

# https://poe.com/api/models

In [5]:
current_tools = None


def set_tools(tools=None):
    global current_tools
    current_tools = tools

In [6]:
class Param:
    def __init__(
        self,
        param_type: str | np.ndarray | dict,
        param_desc: str,
        is_required: bool = True,
        *,
        dtype="number",
        enum=None,
    ):
        # Discourage 'object'. Prefer flat.
        self.param_type = param_type
        self.param_desc = param_desc
        self.is_required = is_required
        self.enum = enum
        self.dtype = dtype

    def to_dict(self):
        schema = {
            "description": self.param_desc,
        }

        if self.enum is not None:
            if self.param_type != "string":
                raise ValueError("enum is only allowed for type 'string'")
            schema["enum"] = self.enum

        if isinstance(self.param_type, str):
            if self.param_type == "array":
                schema["type"] = "array"
                schema["items"] = {"type": self.dtype}
            elif self.param_type == "matrix":
                schema["type"] = "array"
                schema["items"] = {"type": "array", "items": {"type": self.dtype}}
            elif self.param_type in ["string", "number", "boolean", "integer"]:
                schema["type"] = self.param_type
            else:
                raise ValueError(f"Unsupported param type: {self.param_type}")
        else:
            raise ValueError(f"Unsupported param type: {self.param_type}")

        return schema


def infer_scalar_type(x):
    if isinstance(x, str):
        return "string"
    elif isinstance(x, bool):
        return "boolean"
    elif isinstance(x, int):
        return "integer"
    elif isinstance(x, float):
        return "number"
    else:
        raise ValueError(f"Unsupported type: {type(x)}")


class ParamSpec:
    def __init__(self, params: dict[str, Param]):
        self.params = params

    def to_dict(self):
        required = [name for name, param in self.params.items() if param.is_required]
        return {
            "type": "object",
            "properties": {
                name: param.to_dict() for name, param in self.params.items()
            },
            "required": required,
            "additionalProperties": False,
        }


ParamSpec(
    {
        "distance_matrix": Param(
            "matrix",
            "2D N*N array of distances between nodes (e.g.: [[0.0, 1.1, 2.2], [1.3, 0.0, 1.4], [2.5, 3.6, 0.0]])",
        ),
    }
).to_dict()

{'type': 'object',
 'properties': {'distance_matrix': {'description': '2D N*N array of distances between nodes (e.g.: [[0.0, 1.1, 2.2], [1.3, 0.0, 1.4], [2.5, 3.6, 0.0]])',
   'type': 'array',
   'items': {'type': 'array', 'items': {'type': 'number'}}}},
 'required': ['distance_matrix'],
 'additionalProperties': False}

In [7]:
from typing import Callable


class Tool:
    def __init__(self, fn: Callable, desc: str, params: ParamSpec):
        self.fn = fn
        self.desc = desc
        self.params = params
        self.name = fn.__name__

    def to_dict(self):
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.desc,
                "parameters": self.params.to_dict(),
            },
        }

In [8]:
tools_solvers: list[Tool] = []

In [9]:
import algorithms

tools_solvers.append(
    Tool(
        algorithms.nearest_neighbor_vrp,
        "Solve VRP using nearest neighbor heuristic",
        ParamSpec(
            {
                "distance_matrix": Param(
                    "matrix",
                    "2D N*N array of distances between nodes (e.g.: [[0.0, 1.1, 2.2], [1.3, 0.0, 1.4], [2.5, 3.6, 0.0]])",
                ),
                "demands": Param(
                    "array",
                    "N-length array of demands for each node (e.g.: [1.0, 1.1, 1.2])",
                ),
                "vehicle_capacity": Param(
                    "number", "Capacity of the vehicle (e.g.: 3.0)"
                ),
            }
        ),
    )
)

tools_solvers[0].to_dict()

{'type': 'function',
 'function': {'name': 'nearest_neighbor_vrp',
  'description': 'Solve VRP using nearest neighbor heuristic',
  'parameters': {'type': 'object',
   'properties': {'distance_matrix': {'description': '2D N*N array of distances between nodes (e.g.: [[0.0, 1.1, 2.2], [1.3, 0.0, 1.4], [2.5, 3.6, 0.0]])',
     'type': 'array',
     'items': {'type': 'array', 'items': {'type': 'number'}}},
    'demands': {'description': 'N-length array of demands for each node (e.g.: [1.0, 1.1, 1.2])',
     'type': 'array',
     'items': {'type': 'number'}},
    'vehicle_capacity': {'description': 'Capacity of the vehicle (e.g.: 3.0)',
     'type': 'number'}},
   'required': ['distance_matrix', 'demands', 'vehicle_capacity'],
   'additionalProperties': False}}}

In [10]:
def llm(
    prompt: str = None,
    *,
    tools: list[Tool] = None,
    model="grok-4.1-fast-reasoning",
    messages: list[dict] = None,
    max_history=10,
    max_recent=5,
):
    if messages is None:
        messages = [
            {
                "role": "system",
                "content": """You are LogisticsBot — the official AI assistant for logistics managers at a trucking/fleet company. 
Your job is to help non-technical managers create, modify, and improve daily truck delivery plans using Optimization Tools.""",
            }
        ]
    else:
        messages = messages.copy()

    if len(messages) > max_history:
        cut_point = max(1, len(messages) - max_recent)

        while cut_point > 1 and messages[cut_point - 1]["role"] == "tool":
            cut_point -= 1

        system_message = messages[0]
        messages_summarized = messages[1:cut_point]
        if len(messages_summarized) > 0:
            messages_kept = messages[cut_point:]

            print("messages_summarized", messages_summarized)

            summarized_content = summarize(messages_summarized)["content"]
            message = {
                "role": "system",
                "content": f"STATE MEMORY:\n {summarized_content}",
            }
            messages = [system_message] + [message] + messages_kept

    if prompt is not None:
        messages.append({"role": "user", "content": prompt})

    kwargs = {}

    if tools is not None:
        kwargs["tools"] = [tool.to_dict() for tool in tools]
        kwargs["parallel_tool_calls"] = True

    chat = client.chat.completions.create(
        model=model,
        messages=messages,
        **kwargs,
    )

    message = chat.choices[0].message.to_dict()
    messages.append(message)

    if "tool_calls" in message:
        tool_messages = handle_tool_calls(message, tools)
        if len(tool_messages) > 0:
            print(f"Tool messages: {tool_messages}")
            messages.extend(tool_messages)
            messages = llm(
                """Please give me a clear, human-readable summary of the latest tool results.  
Include any important numbers, times, distances, costs, etc.

If it's not enough to answer my question properly (empty, infeasible, wrong, missing stuff, etc.), don't stop — try again with a better query, different tool, relaxed rule, or fallback method.  
Only give up after 2-3 good tries, and then tell me why it didn't work.

Be concise.""",
                tools=tools,
                model=model,
                messages=messages,
            )

    return messages


def handle_tool_calls(message: dict, tools: list[Tool]):
    messages = []
    for tool_call in message["tool_calls"]:
        tool_call_id = tool_call["id"]
        tool_name = tool_call["function"]["name"]
        tool_args = tool_call["function"]["arguments"]

        fn = next((tool.fn for tool in tools if tool.name == tool_name), None)

        if fn is None:
            raise ValueError(f"Function {tool_name} not found in local tools")

        print(f"Calling tool: {tool_name} with args: {tool_args}")
        try:
            result = fn(**json.loads(tool_args))
        except Exception as e:
            result = {"error": str(e)}
        print(f"Result: {result}")
        print("=" * 50)

        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call_id,
                "name": tool_name,
                "content": json.dumps(result),
            }
        )
    return messages


def summarize(messages, model="grok-4.1-fast-non-reasoning"):
    content_to_summarize = []
    for msg in messages:
        if msg["role"] in ("user", "assistant"):
            content_to_summarize.append(f"{msg['role'].upper()}: {msg['content']}")

    history_str = "\n\n".join(content_to_summarize)

    prompt = f"""Summarize this conversation history into a compact STATE MEMORY.
Keep: names, IDs, dates, numbers, locations, exact constraints, decisions, current task status.
Remove: greetings, thanks, chit-chat, repetitions, meta comments.

Use **exactly** this format — nothing else:

**Objective**  
One clear sentence.

**Key Facts**  
• bullet
• bullet

**Progress & Decisions**  
• bullet
• bullet

**Next / Open Questions**  
• bullet

History to summarize:
{history_str}"""

    messages = llm(
        prompt,
        messages=[
            {"role": "system", "content": "You are a very concise history summarizer."},
        ],
        model=model,
        max_recent=9999,
    )

    return messages[-1]

In [11]:
dmat_df = pd.read_csv("./data/dmatrix.csv", index_col=False, header=None)
dmat = dmat_df.to_numpy()

In [12]:
messages = None

In [13]:
prompt = f"""
distance matrix: {dmat}
  
all demands are 1, vehicle capacity is 3
list the routes found and show the total distance
"""
messages = llm(prompt, tools=tools_solvers, messages=messages)
message = messages[-1]

Calling tool: nearest_neighbor_vrp with args: {"distance_matrix": [[0.0, 34705.0, 33927.0, 35869.0, 33969.0, 36659.0, 35376.0, 35604.0, 33347.0, 34911.0, 34377.0, 35519.0, 33289.0, 33357.0, 34657.0, 33999.0, 33657.0, 35621.0, 34165.0], [32978.0, 0.0, 3412.0, 1842.0, 2579.0, 2633.0, 1280.0, 1577.0, 832.0, 883.0, 692.0, 1492.0, 772.0, 841.0, 2151.0, 2166.0, 1880.0, 1594.0, 323.0], [33872.0, 1842.0, 0.0, 1033.0, 655.0, 3117.0, 1094.0, 1202.0, 1398.0, 1523.0, 1510.0, 918.0, 1339.0, 1253.0, 1973.0, 188.0, 2340.0, 1287.0, 1965.0], [34493.0, 2531.0, 1473.0, 0.0, 1522.0, 2445.0, 1099.0, 530.0, 2334.0, 851.0, 2553.0, 903.0, 2274.0, 2343.0, 2496.0, 1109.0, 3382.0, 492.0, 2133.0], [33043.0, 1801.0, 958.0, 1470.0, 0.0, 3554.0, 1532.0, 1639.0, 1357.0, 1961.0, 1469.0, 1356.0, 1297.0, 1212.0, 1931.0, 619.0, 2298.0, 1725.0, 1924.0], [33344.0, 1736.0, 3778.0, 2113.0, 3821.0, 0.0, 966.0, 1848.0, 1184.0, 1154.0, 1404.0, 1763.0, 1126.0, 1194.0, 2455.0, 2569.0, 2234.0, 1865.0, 960.0], [32593.0, 1350.0, 194

In [14]:
print(message["content"])

### Routes (6 vehicles, depot=0):
1. 0 → 12 → 8 → 13 → 0  
2. 0 → 16 → 18 → 1 → 0  
3. 0 → 2 → 15 → 11 → 0  
4. 0 → 4 → 10 → 9 → 0  
5. 0 → 14 → 3 → 17 → 0  
6. 0 → 6 → 7 → 5 → 0  

**Total distance: 417,875**


In [15]:
prompt = f"""
what is the exact returned value of the tool call in the previous message? and the tool name?
I dont want a summary, just the raw value
"""
messages = llm(prompt, tools=tools_solvers, messages=messages)
message = messages[-1]
print(message["content"])
messages

**Tool name:** nearest_neighbor_vrp

**Raw returned value:** [[[0, 12, 8, 13, 0], [0, 16, 18, 1, 0], [0, 2, 15, 11, 0], [0, 4, 10, 9, 0], [0, 14, 3, 17, 0], [0, 6, 7, 5, 0]], 417875.0]


[{'role': 'system',
  'content': 'You are LogisticsBot — the official AI assistant for logistics managers at a trucking/fleet company. \nYour job is to help non-technical managers create, modify, and improve daily truck delivery plans using Optimization Tools.'},
 {'role': 'user',
  'content': '\ndistance matrix: [[    0. 34705. 33927. 35869. 33969. 36659. 35376. 35604. 33347. 34911.\n  34377. 35519. 33289. 33357. 34657. 33999. 33657. 35621. 34165.]\n [32978.     0.  3412.  1842.  2579.  2633.  1280.  1577.   832.   883.\n    692.  1492.   772.   841.  2151.  2166.  1880.  1594.   323.]\n [33872.  1842.     0.  1033.   655.  3117.  1094.  1202.  1398.  1523.\n   1510.   918.  1339.  1253.  1973.   188.  2340.  1287.  1965.]\n [34493.  2531.  1473.     0.  1522.  2445.  1099.   530.  2334.   851.\n   2553.   903.  2274.  2343.  2496.  1109.  3382.   492.  2133.]\n [33043.  1801.   958.  1470.     0.  3554.  1532.  1639.  1357.  1961.\n   1469.  1356.  1297.  1212.  1931.   619.  2298.  

In [16]:
prompt = f"""
what if the capacity is doubled.
"""
messages = llm(prompt, tools=tools_solvers, messages=messages)
message = messages[-1]
print(message["content"])
messages

Calling tool: nearest_neighbor_vrp with args: {"distance_matrix": [[0.0, 34705.0, 33927.0, 35869.0, 33969.0, 36659.0, 35376.0, 35604.0, 33347.0, 34911.0, 34377.0, 35519.0, 33289.0, 33357.0, 34657.0, 33999.0, 33657.0, 35621.0, 34165.0], [32978.0, 0.0, 3412.0, 1842.0, 2579.0, 2633.0, 1280.0, 1577.0, 832.0, 883.0, 692.0, 1492.0, 772.0, 841.0, 2151.0, 2166.0, 1880.0, 1594.0, 323.0], [33872.0, 1842.0, 0.0, 1033.0, 655.0, 3117.0, 1094.0, 1202.0, 1398.0, 1523.0, 1510.0, 918.0, 1339.0, 1253.0, 1973.0, 188.0, 2340.0, 1287.0, 1965.0], [34493.0, 2531.0, 1473.0, 0.0, 1522.0, 2445.0, 1099.0, 530.0, 2334.0, 851.0, 2553.0, 903.0, 2274.0, 2343.0, 2496.0, 1109.0, 3382.0, 492.0, 2133.0], [33043.0, 1801.0, 958.0, 1470.0, 0.0, 3554.0, 1532.0, 1639.0, 1357.0, 1961.0, 1469.0, 1356.0, 1297.0, 1212.0, 1931.0, 619.0, 2298.0, 1725.0, 1924.0], [33344.0, 1736.0, 3778.0, 2113.0, 3821.0, 0.0, 966.0, 1848.0, 1184.0, 1154.0, 1404.0, 1763.0, 1126.0, 1194.0, 2455.0, 2569.0, 2234.0, 1865.0, 960.0], [32593.0, 1350.0, 194

[{'role': 'system',
  'content': 'You are LogisticsBot — the official AI assistant for logistics managers at a trucking/fleet company. \nYour job is to help non-technical managers create, modify, and improve daily truck delivery plans using Optimization Tools.'},
 {'role': 'system',
  'content': 'STATE MEMORY:\n **Objective**  \nSolve VRP with 19-node distance matrix (depot 0), all demands=1, vehicle capacity=3; list routes and total distance.\n\n**Key Facts**  \n• 19 nodes, depot=0, demands=1 each, capacity=3 per vehicle.  \n• Distance matrix provided (e.g., [0, 34705, 33927, ...]).\n\n**Progress & Decisions**  \n• 6 routes found: (1) 0-12-8-13-0; (2) 0-16-18-1-0; (3) 0-2-15-11-0; (4) 0-4-10-9-0; (5) 0-14-3-17-0; (6) 0-6-7-5-0.  \n• Total distance: 417,875.\n\n**Next / Open Questions**  \n• None (routes and total distance provided).'},
 {'role': 'user',
  'content': '\nwhat is the exact returned value of the tool call in the previous message? and the tool name?\nI dont want a summary,

In [17]:
prompt = f"""
what if the capacity is further doubled.
"""
messages = llm(prompt, tools=tools_solvers, messages=messages)
message = messages[-1]
print(message["content"])
messages

Calling tool: nearest_neighbor_vrp with args: {"distance_matrix": [[0.0, 34705.0, 33927.0, 35869.0, 33969.0, 36659.0, 35376.0, 35604.0, 33347.0, 34911.0, 34377.0, 35519.0, 33289.0, 33357.0, 34657.0, 33999.0, 33657.0, 35621.0, 34165.0], [32978.0, 0.0, 3412.0, 1842.0, 2579.0, 2633.0, 1280.0, 1577.0, 832.0, 883.0, 692.0, 1492.0, 772.0, 841.0, 2151.0, 2166.0, 1880.0, 1594.0, 323.0], [33872.0, 1842.0, 0.0, 1033.0, 655.0, 3117.0, 1094.0, 1202.0, 1398.0, 1523.0, 1510.0, 918.0, 1339.0, 1253.0, 1973.0, 188.0, 2340.0, 1287.0, 1965.0], [34493.0, 2531.0, 1473.0, 0.0, 1522.0, 2445.0, 1099.0, 530.0, 2334.0, 851.0, 2553.0, 903.0, 2274.0, 2343.0, 2496.0, 1109.0, 3382.0, 492.0, 2133.0], [33043.0, 1801.0, 958.0, 1470.0, 0.0, 3554.0, 1532.0, 1639.0, 1357.0, 1961.0, 1469.0, 1356.0, 1297.0, 1212.0, 1931.0, 619.0, 2298.0, 1725.0, 1924.0], [33344.0, 1736.0, 3778.0, 2113.0, 3821.0, 0.0, 966.0, 1848.0, 1184.0, 1154.0, 1404.0, 1763.0, 1126.0, 1194.0, 2455.0, 2569.0, 2234.0, 1865.0, 960.0], [32593.0, 1350.0, 194

[{'role': 'system',
  'content': 'You are LogisticsBot — the official AI assistant for logistics managers at a trucking/fleet company. \nYour job is to help non-technical managers create, modify, and improve daily truck delivery plans using Optimization Tools.'},
 {'role': 'system',
  'content': 'STATE MEMORY:\n **Objective**  \nQuery exact tool call details from previous message and test impact of doubling capacity on nearest_neighbor_vrp output.\n\n**Key Facts**  \n• Tool name: nearest_neighbor_vrp  \n• Original raw returned value: [[[0,12,8,13,0],[0,16,18,1,0],[0,2,15,11,0],[0,4,10,9,0],[0,14,3,17,0],[0,6,7,5,0]], 417875.0]  \n• New query: double the capacity.\n\n**Progress & Decisions**  \n• Provided exact tool name and raw value as requested.  \n• Capacity doubling query received; response pending.\n\n**Next / Open Questions**  \n• What is the nearest_neighbor_vrp output with doubled capacity?'},
 {'role': 'tool',
  'tool_call_id': 'call_74603085',
  'name': 'nearest_neighbor_vrp'

In [18]:
prompt = f"""
um...lets use capacity of 5.5
"""
messages = llm(prompt, tools=tools_solvers, messages=messages)
message = messages[-1]
print(message["content"])
messages

Calling tool: nearest_neighbor_vrp with args: {"distance_matrix": [[0.0, 34705.0, 33927.0, 35869.0, 33969.0, 36659.0, 35376.0, 35604.0, 33347.0, 34911.0, 34377.0, 35519.0, 33289.0, 33357.0, 34657.0, 33999.0, 33657.0, 35621.0, 34165.0], [32978.0, 0.0, 3412.0, 1842.0, 2579.0, 2633.0, 1280.0, 1577.0, 832.0, 883.0, 692.0, 1492.0, 772.0, 841.0, 2151.0, 2166.0, 1880.0, 1594.0, 323.0], [33872.0, 1842.0, 0.0, 1033.0, 655.0, 3117.0, 1094.0, 1202.0, 1398.0, 1523.0, 1510.0, 918.0, 1339.0, 1253.0, 1973.0, 188.0, 2340.0, 1287.0, 1965.0], [34493.0, 2531.0, 1473.0, 0.0, 1522.0, 2445.0, 1099.0, 530.0, 2334.0, 851.0, 2553.0, 903.0, 2274.0, 2343.0, 2496.0, 1109.0, 3382.0, 492.0, 2133.0], [33043.0, 1801.0, 958.0, 1470.0, 0.0, 3554.0, 1532.0, 1639.0, 1357.0, 1961.0, 1469.0, 1356.0, 1297.0, 1212.0, 1931.0, 619.0, 2298.0, 1725.0, 1924.0], [33344.0, 1736.0, 3778.0, 2113.0, 3821.0, 0.0, 966.0, 1848.0, 1184.0, 1154.0, 1404.0, 1763.0, 1126.0, 1194.0, 2455.0, 2569.0, 2234.0, 1865.0, 960.0], [32593.0, 1350.0, 194

[{'role': 'system',
  'content': 'You are LogisticsBot — the official AI assistant for logistics managers at a trucking/fleet company. \nYour job is to help non-technical managers create, modify, and improve daily truck delivery plans using Optimization Tools.'},
 {'role': 'system',
  'content': "STATE MEMORY:\n **Objective**  \nOptimize Vehicle Routing Problem (VRP) routes by testing increased vehicle capacities, starting from capacity=6.0 covering 18 customers.\n\n**Key Facts**  \n• Capacity=6.0: 3 routes covering 18 customers; total distance 213,982 (49% less than capacity=3.0's 417,875).  \n• Routes: (1) 0→12→8→13→10→1→18→0; (2) 0→16→9→6→14→15→11→0; (3) 0→2→4→3→17→7→5→0.  \n• User requests testing doubled capacity (12.0).\n\n**Progress & Decisions**  \n• Capacity=6.0 solved with 3 efficient routes vs. more vehicles at 3.0.  \n• Awaiting results for capacity=12.0.\n\n**Next / Open Questions**  \n• Compute and summarize VRP results for capacity=12.0 (expect fewer routes/shorter dista